# ***Merge***



**Miêu tả:** Phần Merge được chia thành 4 cell, mỗi cell đảm nhiệm một chức năng riêng để dễ chạy và kiểm soát kết quả. Cell 1 dùng để import các thư viện cần thiết và tạo thư mục output/ nếu chưa tồn tại. Cell 2 xử lý tập train bằng cách đọc train_clean.csv, user_features.csv và review_features.csv, sau đó merge các đặc trưng vào dữ liệu gốc và xuất ra train_merged_features.csv. Cell 3 thực hiện tương tự cho tập test và lưu thành test_merged_features.csv. Cell 4 thực hiện tương tự cho tập validation và lưu thành val_merged_features.csv.

Trong mỗi cell xử lý dữ liệu, logic merge được giữ thống nhất. Trước tiên, code kiểm tra và sửa tên cột ' Trang' thành 'content' nếu file bị parse sai tên cột. Sau đó, chương trình xác định các cột feature mới để tránh thêm trùng những cột đã tồn tại trong dữ liệu gốc. Các đặc trưng cấp người dùng được merge bằng left join theo customer_id, còn các đặc trưng cấp review được merge bằng left join theo content. Nếu cột content trong file review features không unique, code sẽ deduplicate trước khi merge để tránh làm tăng số dòng ngoài ý muốn.

# **Imports & Setup**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

IN = Path("outputs")
IN1 = Path("outputs1")
OUT = Path("outputs2")
OUT.mkdir(exist_ok=True)

# **Merge Train**

In [2]:
print("Đọc files...")
train   = pd.read_csv(IN / "train_clean.csv",      low_memory=False)
user_f  = pd.read_csv(IN1 / "user_features.csv",    low_memory=False)
review_f= pd.read_csv(IN1 / "review_features.csv",  low_memory=False)

if ' Trang' in review_f.columns:
    review_f = review_f.rename(columns={' Trang': 'content'})
elif 'content' not in review_f.columns:
    print(f"Warning: 'content' column not found. Available columns: {review_f.columns.tolist()}")

print(f"  train_clean    : {train.shape[0]:,} rows × {train.shape[1]} cols")
print(f"  user_features  : {user_f.shape[0]:,} rows × {user_f.shape[1]} cols")
print(f"  review_features: {review_f.shape[0]:,} rows × {review_f.shape[1]} cols")

USER_KEY, REVIEW_KEY = "customer_id", "content"

user_new_cols   = [c for c in user_f.columns   if c != USER_KEY   and c not in train.columns]
review_new_cols = [c for c in review_f.columns if c not in (REVIEW_KEY, USER_KEY, "label")
                   and c not in train.columns]

print(f"\n  Cột mới từ user_features   ({len(user_new_cols)}): {user_new_cols}")
print(f"  Cột mới từ review_features ({len(review_new_cols)}): {review_new_cols}")

train = train.merge(user_f[[USER_KEY] + user_new_cols], on=USER_KEY, how="left")
print(f"\nSau merge user   : {train.shape[0]:,} rows × {train.shape[1]} cols")

if not review_f[REVIEW_KEY].is_unique:
    print(f"  Warning: '{REVIEW_KEY}' not unique → deduplicating...")
    review_f_to_merge = review_f.drop_duplicates(subset=[REVIEW_KEY]).copy()
else:
    review_f_to_merge = review_f.copy()

train = train.merge(review_f_to_merge[[REVIEW_KEY] + review_new_cols], on=REVIEW_KEY, how="left")
print(f"Sau merge review : {train.shape[0]:,} rows × {train.shape[1]} cols")

train.to_csv(OUT / "train_merged_features.csv", index=False)
print(f"Saved: train_merged_features.csv\n")

Đọc files...
  train_clean    : 12 rows × 18 cols
  user_features  : 10 rows × 15 cols
  review_features: 6 rows × 25 cols

  Cột mới từ user_features   (14): ['date_min', 'date_max', 'active_days', 'review_count_total', 'reviews_per_day', 'time_gap_std', 'max_reviews_in_one_day', 'account_age_days', 'rating_variance', 'percent_extreme_rating', 'single_brand_focus_ratio', 'review_length_variance', 'avatar_default', 'label']
  Cột mới từ review_features (7): ['promo_keyword_ratio', 'external_link_flag', 'contact_info_flag', 'excessive_punctuation', 'capitalization_ratio', 'emoji_density', 'word_repetition_ratio']

Sau merge user   : 12 rows × 32 cols
Sau merge review : 12 rows × 39 cols
Saved: train_merged_features.csv



# **Merge Test**

In [3]:
print("Đọc files...")
test    = pd.read_csv(IN / "test_clean.csv",           low_memory=False)
user_f  = pd.read_csv(IN1 / "user_features_test.csv",   low_memory=False)
review_f= pd.read_csv(IN1 / "review_features_test.csv", low_memory=False)

if ' Trang' in review_f.columns:
    review_f = review_f.rename(columns={' Trang': 'content'})
elif 'content' not in review_f.columns:
    print(f"Warning: 'content' column not found. Available columns: {review_f.columns.tolist()}")

print(f"  test_clean     : {test.shape[0]:,} rows × {test.shape[1]} cols")
print(f"  user_features  : {user_f.shape[0]:,} rows × {user_f.shape[1]} cols")
print(f"  review_features: {review_f.shape[0]:,} rows × {review_f.shape[1]} cols")

USER_KEY, REVIEW_KEY = "customer_id", "content"

user_new_cols   = [c for c in user_f.columns   if c != USER_KEY   and c not in test.columns]
review_new_cols = [c for c in review_f.columns if c not in (REVIEW_KEY, USER_KEY, "label")
                   and c not in test.columns]

print(f"\n  Cột mới từ user_features   ({len(user_new_cols)}): {user_new_cols}")
print(f"  Cột mới từ review_features ({len(review_new_cols)}): {review_new_cols}")

test = test.merge(user_f[[USER_KEY] + user_new_cols], on=USER_KEY, how="left")
print(f"\nSau merge user   : {test.shape[0]:,} rows × {test.shape[1]} cols")

if not review_f[REVIEW_KEY].is_unique:
    print(f"  Warning: '{REVIEW_KEY}' not unique → deduplicating...")
    review_f_to_merge = review_f.drop_duplicates(subset=[REVIEW_KEY]).copy()
else:
    review_f_to_merge = review_f.copy()

test = test.merge(review_f_to_merge[[REVIEW_KEY] + review_new_cols], on=REVIEW_KEY, how="left")
print(f"Sau merge review : {test.shape[0]:,} rows × {test.shape[1]} cols")

test.to_csv(OUT / "test_merged_features.csv", index=False)
print(f"Saved: test_merged_features.csv\n")

Đọc files...
  test_clean     : 3 rows × 18 cols
  user_features  : 3 rows × 15 cols
  review_features: 2 rows × 25 cols

  Cột mới từ user_features   (14): ['date_min', 'date_max', 'active_days', 'review_count_total', 'reviews_per_day', 'time_gap_std', 'max_reviews_in_one_day', 'account_age_days', 'rating_variance', 'percent_extreme_rating', 'single_brand_focus_ratio', 'review_length_variance', 'avatar_default', 'label']
  Cột mới từ review_features (7): ['promo_keyword_ratio', 'external_link_flag', 'contact_info_flag', 'excessive_punctuation', 'capitalization_ratio', 'emoji_density', 'word_repetition_ratio']

Sau merge user   : 3 rows × 32 cols
Sau merge review : 3 rows × 39 cols
Saved: test_merged_features.csv



# **Merge Val**

In [4]:
print("Đọc files...")
val     = pd.read_csv(IN / "val_clean.csv",           low_memory=False)
user_f  = pd.read_csv(IN1 / "user_features_val.csv",   low_memory=False)
review_f= pd.read_csv(IN1 / "review_features_val.csv", low_memory=False)

if ' Trang' in review_f.columns:
    review_f = review_f.rename(columns={' Trang': 'content'})
elif 'content' not in review_f.columns:
    print(f"Warning: 'content' column not found. Available columns: {review_f.columns.tolist()}")

print(f"  val_clean      : {val.shape[0]:,} rows × {val.shape[1]} cols")
print(f"  user_features  : {user_f.shape[0]:,} rows × {user_f.shape[1]} cols")
print(f"  review_features: {review_f.shape[0]:,} rows × {review_f.shape[1]} cols")

USER_KEY, REVIEW_KEY = "customer_id", "content"

user_new_cols   = [c for c in user_f.columns   if c != USER_KEY   and c not in val.columns]
review_new_cols = [c for c in review_f.columns if c not in (REVIEW_KEY, USER_KEY, "label")
                   and c not in val.columns]

print(f"\n  Cột mới từ user_features   ({len(user_new_cols)}): {user_new_cols}")
print(f"  Cột mới từ review_features ({len(review_new_cols)}): {review_new_cols}")

val = val.merge(user_f[[USER_KEY] + user_new_cols], on=USER_KEY, how="left")
print(f"\nSau merge user   : {val.shape[0]:,} rows × {val.shape[1]} cols")

if not review_f[REVIEW_KEY].is_unique:
    print(f"  Warning: '{REVIEW_KEY}' not unique → deduplicating...")
    review_f_to_merge = review_f.drop_duplicates(subset=[REVIEW_KEY]).copy()
else:
    review_f_to_merge = review_f.copy()

val = val.merge(review_f_to_merge[[REVIEW_KEY] + review_new_cols], on=REVIEW_KEY, how="left")
print(f"Sau merge review : {val.shape[0]:,} rows × {val.shape[1]} cols")

val.to_csv(OUT / "val_merged_features.csv", index=False)
print(f"Saved: val_merged_features.csv\n")

Đọc files...
  val_clean      : 2 rows × 18 cols
  user_features  : 2 rows × 15 cols
  review_features: 1 rows × 25 cols

  Cột mới từ user_features   (14): ['date_min', 'date_max', 'active_days', 'review_count_total', 'reviews_per_day', 'time_gap_std', 'max_reviews_in_one_day', 'account_age_days', 'rating_variance', 'percent_extreme_rating', 'single_brand_focus_ratio', 'review_length_variance', 'avatar_default', 'label']
  Cột mới từ review_features (7): ['promo_keyword_ratio', 'external_link_flag', 'contact_info_flag', 'excessive_punctuation', 'capitalization_ratio', 'emoji_density', 'word_repetition_ratio']

Sau merge user   : 2 rows × 32 cols
Sau merge review : 2 rows × 39 cols
Saved: val_merged_features.csv



In [5]:
import pandas as pd

df1 = pd.read_csv('outputs2/train_merged_features.csv')
df1.isnull().sum()
df1.shape

(12, 39)